In [326]:
pip install pandas sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [327]:
import pandas as pd

In [328]:
faturas_csv = ['DadosFaturas/Fatura_2025-03-20.csv', 'DadosFaturas/Fatura_2025-04-20.csv', 'DadosFaturas/Fatura_2025-05-20.csv', 
               'DadosFaturas/Fatura_2025-06-20.csv', 'DadosFaturas/Fatura_2025-07-20.csv', 'DadosFaturas/Fatura_2025-08-20.csv',
               'DadosFaturas/Fatura_2025-09-20.csv', 'DadosFaturas/Fatura_2025-10-20.csv', 'DadosFaturas/Fatura_2025-11-20.csv',
               'DadosFaturas/Fatura_2025-12-20.csv', 'DadosFaturas/Fatura_2026-01-20.csv', 'DadosFaturas/Fatura_2026-02-20.csv']

dados_faturas_list = []

for f in faturas_csv:
    fatura = pd.read_csv(f, sep=';')
    dados_faturas_list.append(fatura)

dados_faturas = pd.concat(dados_faturas_list)

In [329]:
dados_faturas.rename(columns={
    'Nome no Cartão': 'nome_cartao', 
    'Final do Cartão': 'final_cartao',
    'Data de Compra': 'data_completa',
    'Categoria': 'nome_categoria', 
    'Descrição': 'nome_estabelecimento',
    'Valor (em R$)': 'valor_brl', 
    'Valor (em US$)': 'valor_usd', 
    'Parcela': 'parcela_texto', 
    'Cotação (em R$)': 'cotacao_brl'
}, inplace=True)

In [330]:
print("Linhas carregadas:", len(dados_faturas))

Linhas carregadas: 1945


In [331]:
# DIM titular

titulares = dados_faturas[["nome_cartao", "final_cartao"]].drop_duplicates().copy()
titulares["id_titular"] = range(1, len(titulares) + 1)
dados_faturas = dados_faturas.merge(titulares, on=["nome_cartao", "final_cartao"], how="left")
dados_faturas = dados_faturas.drop(columns=["nome_cartao", "final_cartao"])

In [332]:
print("titulares:", len(titulares))

titulares: 5


In [333]:
# DIM categoria

dados_faturas["nome_categoria"] = dados_faturas["nome_categoria"].replace("-", "Não categorizado")

categorias = dados_faturas[["nome_categoria"]].drop_duplicates().copy()
categorias["id_categoria"] = range(1, len(categorias) + 1)
dados_faturas = dados_faturas.merge(categorias, on=["nome_categoria"], how="left")
dados_faturas = dados_faturas.drop(columns=["nome_categoria"])

In [334]:
print("Categorias:", len(categorias))

Categorias: 36


In [335]:
#DIM estabelecimentos

estabelecimentos = dados_faturas[["nome_estabelecimento"]].drop_duplicates().copy()
estabelecimentos["id_estabelecimento"] = range(1, len(estabelecimentos) + 1)
dados_faturas = dados_faturas.merge(estabelecimentos, on=["nome_estabelecimento"], how="left")
dados_faturas = dados_faturas.drop(columns=["nome_estabelecimento"])

In [336]:
print(len(estabelecimentos))

426


In [337]:
# DIM data_compra

from datetime import datetime

datas_compra = dados_faturas[["data_completa"]].drop_duplicates().copy()
datas_compra["id_data"] = range(1, len(datas_compra) + 1)

datas_compra["data_completa"] = pd.to_datetime(datas_compra["data_completa"], format="%d/%m/%Y")

datas_compra["dia"] = datas_compra["data_completa"].dt.day
datas_compra["mes"] = datas_compra["data_completa"].dt.month
datas_compra["ano"] = datas_compra["data_completa"].dt.year
datas_compra["trimestre"] = datas_compra["data_completa"].dt.quarter
datas_compra["dia_semana"] = datas_compra["data_completa"].dt.dayofweek

dados_faturas["data_completa"] = pd.to_datetime(dados_faturas["data_completa"], format="%d/%m/%Y")
dados_faturas = dados_faturas.merge(datas_compra, on=["data_completa"], how="left")
dados_faturas = dados_faturas.drop(columns=["data_completa", "dia", "mes", "ano", "trimestre", "dia_semana"])

In [338]:
print("Datas:", len(datas_compra))

Datas: 343


In [339]:
# Tabela fato transacao

transacoes = dados_faturas[["id_titular", "id_data", "id_categoria", "id_estabelecimento", 
                            "valor_usd", "valor_brl", "cotacao_brl", "parcela_texto"]].copy()
transacoes["id_transacao"] = range(1, len(transacoes) + 1)

transacoes.loc[transacoes["parcela_texto"] == "Única", "parcela_texto"] = "1/1"
transacoes["num_parcela"] = transacoes["parcela_texto"].str.split("/").str[0]
transacoes["total_parcelas"] = transacoes["parcela_texto"].str.split("/").str[1]

In [340]:
print("transacoes:", len(transacoes))
print("duplicatas:", transacoes.duplicated().sum())

transacoes: 1945
duplicatas: 0


In [341]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port= os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

In [342]:
#Verifica se todos os IDs de titular nas transações existem na tabela titulares.
print(transacoes["id_titular"].isin(titulares["id_titular"]).all())

#Confere se todas as datas referenciadas existem na tabela datas.
print(transacoes["id_data"].isin(datas_compra["id_data"]).all())

#Confere se todas as categorias referenciadas existem na tabela categorias
print(transacoes["id_categoria"].isin(categorias["id_categoria"]).all())

#Confere se todos os estabelecimentos referenciados existem na tabela estabelecimentos
print(transacoes["id_estabelecimento"].isin(estabelecimentos["id_estabelecimento"]).all())

#Se aparecer número > 0, significa que existem transações duplicadas.
print(transacoes.duplicated().sum())

#Saída esperada
#True
#True
#True
#True
#0

True
True
True
True
0


In [343]:
titulares.to_sql("titulares", engine, if_exists="append", index=False)

5

In [344]:
categorias.to_sql("categorias", engine, if_exists="append", index=False)

36

In [345]:
datas_compra.to_sql("datas_compra", engine, if_exists="append", index=False)

343

In [346]:
estabelecimentos.to_sql("estabelecimentos", engine, if_exists="append", index=False)

426

In [347]:
transacoes.to_sql("transacoes", engine, if_exists="append", index=False)

945

In [348]:
from sqlalchemy import text

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM titulares")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM categorias")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM datas_compra")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM estabelecimentos")).fetchone())
    print(conn.execute(text("SELECT COUNT(*) FROM transacoes")).fetchone())

(5,)
(36,)
(343,)
(426,)
(1945,)
